In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into memory (stored for later steps)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")

# Timezone for UTC+8
tz = pytz.timezone("Asia/Shanghai")

# Overall min/max timestamps (as epoch and localized datetimes)
metric_min_ts = int(metric_df['timestamp'].min()) if not metric_df.empty else None
metric_max_ts = int(metric_df['timestamp'].max()) if not metric_df.empty else None
trace_min_ts = int(trace_df['timestamp'].min()) if not trace_df.empty else None
trace_max_ts = int(trace_df['timestamp'].max()) if not trace_df.empty else None

metric_min_ts_local = pd.to_datetime(metric_min_ts, unit='s').tz_localize(tz) if metric_min_ts is not None else None
metric_max_ts_local = pd.to_datetime(metric_max_ts, unit='s').tz_localize(tz) if metric_max_ts is not None else None
trace_min_ts_local = pd.to_datetime(trace_min_ts, unit='s').tz_localize(tz) if trace_min_ts is not None else None
trace_max_ts_local = pd.to_datetime(trace_max_ts, unit='s').tz_localize(tz) if trace_max_ts is not None else None

# Aggregation functions for percentiles
def p50(x): return x.quantile(0.50)
def p95(x): return x.quantile(0.95)
def p99(x): return x.quantile(0.99)

# Aggregate metric.csv by (cmdb_id, kpi_name)
metric_agg = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value','count'),
        min_value=('value','min'),
        p50=('value', p50),
        p95=('value', p95),
        p99=('value', p99),
        max_value=('value','max')
    )
)

# Aggregate trace.csv by (cmdb_id, trace_name)
trace_agg = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value','count'),
        min_value=('value','min'),
        p50=('value', p50),
        p95=('value', p95),
        p99=('value', p99),
        max_value=('value','max')
    )
)

# Keep outputs concise: sort by count desc and limit to top 50 rows
metric_agg_top = metric_agg.sort_values(['count','cmdb_id','kpi_name'], ascending=[False, True, True]).head(50)
trace_agg_top = trace_agg.sort_values(['count','cmdb_id','trace_name'], ascending=[False, True, True]).head(50)

# Final displayed variables (compact)
metric_agg_top, metric_min_ts_local, metric_max_ts_local, trace_agg_top, trace_min_ts_local, trace_max_ts_local

```
Out[1]:
```
metric_groups = metric_agg.shape[0]
metric_count_min, metric_count_max = int(metric_agg['count'].min()), int(metric_agg['count'].max())
trace_groups = trace_agg.shape[0]
trace_count_min, trace_count_max = int(trace_agg['count'].min()), int(trace_agg['count'].max())

summary = (
    f"Summary of aggregates computed:\n"
    f"- metric.csv: {metric_groups} unique (cmdb_id, kpi_name) groups aggregated (columns: count, min, p50, p95, p99, max).\n"
    f"  • Displayed: top 50 groups by count.\n"
    f"  • Timestamp range (UTC+8): {metric_min_ts_local} to {metric_max_ts_local}.\n"
    f"  • Group counts observed: {metric_count_min} to {metric_count_max} (many groups show counts 60–61).\n"
    f"  • Example top groups in the display: db_001 CPU_free_pct (count 61, p50 ≈ 98.71), db_003 CPU_free_pct (count 61, p50 ≈ 98.83).\n\n"
    f"- trace.csv: {trace_groups} unique (cmdb_id, trace_name) groups aggregated (columns: count, min, p50, p95, p99, max).\n"
    f"  • Displayed: top 50 groups by count.\n"
    f"  • Timestamp range (UTC+8): {trace_min_ts_local} to {trace_max_ts_local}.\n"
    f"  • Group counts observed: {trace_count_min} to {trace_count_max} (most groups have count 30).\n"
    f"  • Example top groups in the display: docker_001 trace.from_os_022.remoteprocess.duration_mean (count 30, p50 ≈ 0.763),\n"
    f"    docker_001 trace.from_os_022.remoteprocess.duration_p95 (count 30, p50 ≈ 2.075).\n\n"
    f"Notes: Full loaded tables are retained in kernel variables metric_df and trace_df; aggregated tables are in metric_agg and trace_agg. "
    f"Use those for further filtering or deeper analysis."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id             kpi_name  count  min_value        p50        p95        p99  max_value
5    db_001         CPU_free_pct     61  96.644584  98.710390  98.928224  98.977735  99.004524
97   db_003         CPU_free_pct     61  96.879022  98.831692  99.273341  99.391893  99.406910
294  db_007        MEM_real_util     61  83.520000  83.560000  83.570000  83.570000  83.570000
327  db_008         CPU_free_pct     61  95.805982  97.272608  97.822829  97.909751  97.950036
966  os_006           Send_total     61   0.000304   0.000369   0.000476   0.000487   0.000497
..      ...                  ...    ...        ...        ...        ...        ...        ...
47   db_002                 AIOS     60   0.000000   0.000000   0.000000   0.004100   0.010000
48   db_002                  AWS     60   0.000000   0.000000   0.020000   0.024100   0.030000
51   db_002         CPU_free_pct     60  96.829160  98.774531  99.166530  99.269495  99.332782
52   db_002         Call_Per_Sec     60  17.100000  17.400000  17.705000  21.777000  27.500000
53   db_002  DFParaWrite_Per_Sec     60   0.130000   0.435000   0.966000   3.898100   4.140000

[50 rows x 8 columns], Timestamp('2020-05-29 20:30:00+0800', tz='Asia/Shanghai'), Timestamp('2020-05-29 21:30:00+0800', tz='Asia/Shanghai'),        cmdb_id                                       trace_name  count   min_value          p50          p95          p99    max_value
0   docker_001    trace.from_os_022.remoteprocess.duration_mean     30    0.248750     0.763078     1.028691     1.088790     1.101521
1   docker_001     trace.from_os_022.remoteprocess.duration_p95     30    0.453300     2.075050     2.600455     2.928340     3.031000
2   docker_001       trace.from_os_022.remoteprocess.error_rate     30    0.000000     0.000000     0.000000     0.000000     0.000000
3   docker_001        trace.from_os_022.remoteprocess.row_count     30   33.000000   136.000000   159.500000   170.390000   173.000000
4   docker_001                     trace.self.csf.duration_mean     30    0.029068     0.062579     0.075285     0.079795     0.081481
..         ...                                              ...    ...         ...          ...          ...          ...          ...
45  docker_002                    trace.self.local.duration_p95     30    0.081050     0.094000     0.810910     0.811000     0.811000
46  docker_002                      trace.self.local.error_rate     30    0.000000     0.000000     0.000000     0.000000     0.000000
47  docker_002                       trace.self.local.row_count     30  501.000000  1988.000000  2212.000000  2271.640000  2296.000000
48  docker_002  trace.to_docker_007.remoteprocess.duration_mean     30    0.023256     0.025780     0.026750     0.026825     0.026841
49  docker_002   trace.to_docker_007.remoteprocess.duration_p95     30    0.032000     0.036550     0.040165     0.040371     0.040400

[50 rows x 8 columns], Timestamp('2020-05-29 21:00:00+0800', tz='Asia/Shanghai'), Timestamp('2020-05-29 21:29:00+0800', tz='Asia/Shanghai'))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_df and trace_df loaded in kernel

# 1) Compute global P95 and P5 per group using full series (rule: before filtering)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# Totals of groups checked
metric_groups_total = metric_thresholds.shape[0]
trace_groups_total = trace_thresholds.shape[0]

# 2) Filter to incident window: May 30, 2020, 05:00 to 05:30 (UTC+8) inclusive
start_local = pd.Timestamp('2020-05-30 05:00:00', tz=tz)
end_local   = pd.Timestamp('2020-05-30 05:30:00', tz=tz)

# Convert local window to UTC epoch seconds for comparison (assume timestamps are epoch seconds UTC)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

metric_df_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()
trace_df_window  = trace_df[(trace_df['timestamp']  >= start_epoch) & (trace_df['timestamp']  <= end_epoch)].copy()

# 3) For each group compute counts in-window, count_high, count_low, earliest_anom_ts (value >= p95 or <= p5)
# Merge thresholds into windowed rows to evaluate anomalies per row
metric_window_merged = metric_df_window.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')
metric_window_merged['is_high'] = metric_window_merged['value'] >= metric_window_merged['p95']
metric_window_merged['is_low']  = metric_window_merged['value'] <= metric_window_merged['p5']
metric_window_merged['is_anom'] = metric_window_merged['is_high'] | metric_window_merged['is_low']

# Aggregate per group for metric
metric_anom_groups = (
    metric_window_merged
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        group_count_in_window=('value','count'),
        count_high=('is_high','sum'),
        count_low=('is_low','sum'),
        earliest_anom_ts_raw=('timestamp', lambda s: s[metric_window_merged.loc[s.index, 'is_anom']].min() if s.size>0 else pd.NA)
    )
)

# Map earliest_anom_ts_raw to timezone-aware timestamps (UTC -> Asia/Shanghai), null if NA
def to_local_ts(x):
    if pd.isna(x):
        return pd.NaT
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)

metric_anom_groups['earliest_anom_ts'] = metric_anom_groups['earliest_anom_ts_raw'].apply(to_local_ts)
metric_anom_groups.drop(columns=['earliest_anom_ts_raw'], inplace=True)

# Join p95/p5 for display
metric_anoms = metric_anom_groups.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Filter only groups with anomalies (>0)
metric_anoms = metric_anoms[(metric_anoms['count_high'] + metric_anoms['count_low']) > 0].copy()

# Sort as requested and limit to 20 rows, select columns
metric_anoms = metric_anoms.assign(total_anoms=(metric_anoms['count_high'] + metric_anoms['count_low']))
metric_anoms = metric_anoms.sort_values(['total_anoms','earliest_anom_ts'], ascending=[False, True])
metric_anoms = metric_anoms[['cmdb_id','kpi_name','group_count_in_window','count_high','count_low','earliest_anom_ts','p95','p5']].head(20)

metric_groups_with_anoms = metric_anoms.shape[0]

# ---- Repeat for trace ----
trace_window_merged = trace_df_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
trace_window_merged['is_high'] = trace_window_merged['value'] >= trace_window_merged['p95']
trace_window_merged['is_low']  = trace_window_merged['value'] <= trace_window_merged['p5']
trace_window_merged['is_anom'] = trace_window_merged['is_high'] | trace_window_merged['is_low']

trace_anom_groups = (
    trace_window_merged
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        group_count_in_window=('value','count'),
        count_high=('is_high','sum'),
        count_low=('is_low','sum'),
        earliest_anom_ts_raw=('timestamp', lambda s: s[trace_window_merged.loc[s.index, 'is_anom']].min() if s.size>0 else pd.NA)
    )
)

trace_anom_groups['earliest_anom_ts'] = trace_anom_groups['earliest_anom_ts_raw'].apply(to_local_ts)
trace_anom_groups.drop(columns=['earliest_anom_ts_raw'], inplace=True)

trace_anoms = trace_anom_groups.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
trace_anoms = trace_anoms[(trace_anoms['count_high'] + trace_anoms['count_low']) > 0].copy()
trace_anoms = trace_anoms.assign(total_anoms=(trace_anoms['count_high'] + trace_anoms['count_low']))
trace_anoms = trace_anoms.sort_values(['total_anoms','earliest_anom_ts'], ascending=[False, True])
trace_anoms = trace_anoms[['cmdb_id','trace_name','group_count_in_window','count_high','count_low','earliest_anom_ts','p95','p5']].head(20)

trace_groups_with_anoms = trace_anoms.shape[0]

# Final displayed outputs (compact): the two tables and four counts
metric_anoms, trace_anoms, metric_groups_total, metric_groups_with_anoms, trace_groups_total, trace_groups_with_anoms

```
Out[2]:
```
metric_anom_count = metric_groups_with_anoms
metric_total = metric_groups_total
trace_anom_count = trace_groups_with_anoms
trace_total = trace_groups_total

# Build concise plain-English summary
summary = (
    f"Summary of in-window anomalies (May 30, 2020 05:00–05:30 UTC+8):\n"
    f"- Metrics: checked {metric_total} (cmdb_id,kpi) groups; {metric_anom_count} groups had anomalies in-window.\n"
    f"  • Many flagged metric groups show every in-window sample as anomalous (count_high + count_low == group_count_in_window).\n"
    f"  • Earliest anomalies are typically at 2020-05-30 05:00:00+08:00 for these groups.\n"
    f"  • Examples in the results: db_001 Hang, On_Off_State, Row_Lock; several db_* MEM and SctRead metrics; os_007 Disk_rd_ios, os_020 Disk_io_util.\n\n"
    f"- Traces: checked {trace_total} (cmdb_id,trace_name) groups; {trace_anom_count} groups had anomalies in-window.\n"
    f"  • Many trace groups flagged are error_rate or similar metrics with p95 == p5 == 0, which results in all in-window samples being counted as anomalous.\n"
    f"  • Earliest anomalies for traces are also at 2020-05-30 05:00:00+08:00 in the reported results.\n\n"
    f"Note: Full tables and intermediate DataFrames remain available in the kernel (metric_df, trace_df, metric_thresholds, trace_thresholds, metric_anoms, trace_anoms) for follow-up inspection."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id         kpi_name  group_count_in_window  count_high  count_low          earliest_anom_ts         p95          p5
984      os_007      Disk_rd_ios                     31          31         31 2020-05-30 05:00:00+08:00    0.000000    0.000000
1642     os_020     Disk_io_util                     31          31         31 2020-05-30 05:00:00+08:00    0.000000    0.000000
1800  redis_001       redis_load                     31          31         31 2020-05-30 05:00:00+08:00    0.000000    0.000000
10       db_001             Hang                     30          30         30 2020-05-30 05:00:00+08:00    1.000000    1.000000
21       db_001     On_Off_State                     30          30         30 2020-05-30 05:00:00+08:00    1.000000    1.000000
28       db_001         Row_Lock                     30          30         30 2020-05-30 05:00:00+08:00    0.000000    0.000000
30       db_001  SctRead_Per_Sec                     30          30         30 2020-05-30 05:00:00+08:00    0.000000    0.000000
47       db_002             AIOS                     30          30         30 2020-05-30 05:00:00+08:00    0.000000    0.000000
56       db_002             Hang                     30          30         30 2020-05-30 05:00:00+08:00    1.000000    1.000000
61       db_002        MEM_Total                     30          30         30 2020-05-30 05:00:00+08:00  381.902264  381.902264
67       db_002     On_Off_State                     30          30         30 2020-05-30 05:00:00+08:00    1.000000    1.000000
74       db_002         Row_Lock                     30          30         30 2020-05-30 05:00:00+08:00    0.000000    0.000000
76       db_002  SctRead_Per_Sec                     30          30         30 2020-05-30 05:00:00+08:00    0.000000    0.000000
102      db_003             Hang                     30          30         30 2020-05-30 05:00:00+08:00    1.000000    1.000000
113      db_003     On_Off_State                     30          30         30 2020-05-30 05:00:00+08:00    1.000000    1.000000
120      db_003         Row_Lock                     30          30         30 2020-05-30 05:00:00+08:00    0.000000    0.000000
148      db_004             Hang                     30          30         30 2020-05-30 05:00:00+08:00    1.000000    1.000000
153      db_004        MEM_Total                     30          30         30 2020-05-30 05:00:00+08:00  381.902264  381.902264
159      db_004     On_Off_State                     30          30         30 2020-05-30 05:00:00+08:00    1.000000    1.000000
166      db_004         Row_Lock                     30          30         30 2020-05-30 05:00:00+08:00    0.000000    0.000000,        cmdb_id                                    trace_name  group_count_in_window  count_high  count_low          earliest_anom_ts  p95   p5
2   docker_001    trace.from_os_022.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
6   docker_001                     trace.self.csf.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
10  docker_001               trace.self.flyremote.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
14  docker_001                    trace.self.jdbc.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
18  docker_001                   trace.self.local.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
22  docker_001  trace.to_docker_007.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
26  docker_001  trace.to_docker_008.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
30  docker_002    trace.from_os_022.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
34  docker_002                     trace.self.csf.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
38  docker_002               trace.self.flyremote.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
42  docker_002                    trace.self.jdbc.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
46  docker_002                   trace.self.local.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
50  docker_002  trace.to_docker_007.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
54  docker_002  trace.to_docker_008.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
58  docker_003    trace.from_os_021.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
66  docker_003               trace.self.flyremote.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
70  docker_003                    trace.self.jdbc.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
74  docker_003                   trace.self.local.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
78  docker_003  trace.to_docker_005.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0
82  docker_003  trace.to_docker_006.remoteprocess.error_rate                     30          30         30 2020-05-30 05:00:00+08:00  0.0  0.0, 2060, 20, 208, 20)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: metric_df, metric_thresholds, metric_df_window
# Recreate metric_df_window if missing
try:
    metric_df_window
except NameError:
    start_local = pd.Timestamp('2020-05-30 05:00:00', tz=tz)
    end_local   = pd.Timestamp('2020-05-30 05:30:00', tz=tz)
    start_epoch = int(start_local.tz_convert('UTC').timestamp())
    end_epoch   = int(end_local.tz_convert('UTC').timestamp())
    metric_df_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()

# Ensure thresholds exist
try:
    metric_thresholds
except NameError:
    metric_thresholds = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    )

# Merge thresholds into windowed rows
metric_window_merged = metric_df_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Flags
metric_window_merged['is_high'] = metric_window_merged['value'] >= metric_window_merged['p95']
metric_window_merged['is_low']  = metric_window_merged['value'] <= metric_window_merged['p5']
metric_window_merged['is_anom'] = metric_window_merged['is_high'] | metric_window_merged['is_low']

# Helper to convert epoch -> localized tz timestamp
def to_local_ts_val(x):
    if pd.isna(x):
        return pd.NaT
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)

# Build per-group summary for groups that have any anomaly in-window
rows = []
grouped = metric_window_merged.groupby(['cmdb_id','kpi_name'], sort=False)
for (cmdb_id, kpi_name), g in grouped:
    group_count = int(g.shape[0])
    count_high = int(g['is_high'].sum())
    count_low = int(g['is_low'].sum())
    total_anoms = count_high + count_low
    if total_anoms == 0:
        continue

    # earliest anomaly timestamp (min among anomaly rows)
    if g.loc[g['is_anom'],'timestamp'].any():
        earliest_ts_raw = int(g.loc[g['is_anom'],'timestamp'].min())
        earliest_ts = to_local_ts_val(earliest_ts_raw)
    else:
        earliest_ts = pd.NaT

    # longest consecutive anomalous run (consecutive minutes: delta == 60s)
    g_sorted = g.sort_values('timestamp').reset_index(drop=True)
    ts = g_sorted['timestamp'].to_numpy(dtype='int64')
    is_anom = g_sorted['is_anom'].to_numpy(dtype=bool)

    longest_run = 0
    cur_run = 0
    for i in range(len(ts)):
        if is_anom[i]:
            if i>0 and is_anom[i-1] and (ts[i] - ts[i-1] == 60):
                cur_run += 1
            else:
                cur_run = 1
            if cur_run > longest_run:
                longest_run = cur_run
        else:
            cur_run = 0

    # max_abs_deviation for anomaly rows (use p95 for highs and p5 for lows)
    max_dev = 0.0
    had_dev = False
    high_rows = g_sorted[g_sorted['is_high']]
    if not high_rows.empty:
        dev_high = (high_rows['value'] - high_rows['p95']).abs().max()
        if pd.notna(dev_high):
            max_dev = max(max_dev, float(dev_high))
            had_dev = True
    low_rows = g_sorted[g_sorted['is_low']]
    if not low_rows.empty:
        dev_low = (low_rows['value'] - low_rows['p5']).abs().max()
        if pd.notna(dev_low):
            max_dev = max(max_dev, float(dev_low))
            had_dev = True
    max_abs_deviation = float(max_dev) if had_dev else float('nan')

    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'group_count_in_window': group_count,
        'count_high': count_high,
        'count_low': count_low,
        'anomaly_coverage': (total_anoms / group_count) if group_count>0 else 0.0,
        'earliest_anom_ts': earliest_ts,
        'longest_consecutive_anom_len': int(longest_run),
        'max_abs_deviation': max_abs_deviation
    })

metric_group_summary = pd.DataFrame(rows)

# Sort and limit top 20
metric_group_summary = metric_group_summary.sort_values(
    by=['longest_consecutive_anom_len','anomaly_coverage','max_abs_deviation'],
    ascending=[False, False, False]
).reset_index(drop=True)
metric_group_summary_top = metric_group_summary.head(20)

# Component-level aggregation from these groups
if not metric_group_summary.empty:
    comp_agg = metric_group_summary.groupby('cmdb_id', sort=False).apply(
        lambda df: pd.Series({
            'faulty_kpi_count': int(df['kpi_name'].nunique()),
            'earliest_anom_ts': (df['earliest_anom_ts'].min() if df['earliest_anom_ts'].notna().any() else pd.NaT),
            'max_longest_consecutive_anom_len': int(df['longest_consecutive_anom_len'].max()),
            'max_abs_deviation': float(df['max_abs_deviation'].max()),
            'total_anomaly_points': int((df['count_high'] + df['count_low']).sum())
        })
    ).reset_index()
else:
    comp_agg = pd.DataFrame(columns=['cmdb_id','faulty_kpi_count','earliest_anom_ts','max_longest_consecutive_anom_len','max_abs_deviation','total_anomaly_points'])

component_summary_top = comp_agg.sort_values(['faulty_kpi_count','max_longest_consecutive_anom_len'], ascending=[False,False]).head(20)

# Counts for reference
metric_groups_total = int(metric_thresholds.shape[0])
metric_groups_with_anoms = int(metric_group_summary.shape[0])

# Return compact results
metric_group_summary_top, component_summary_top, metric_groups_total, metric_groups_with_anoms

```
Out[3]:
```
summary = (
    "Summary of metric anomaly analysis (May 30, 2020 05:00–05:30 UTC+8):\n"
    "- Per-group results (top examples): many metric groups show every in-window sample flagged as anomalous and have very long consecutive runs.\n"
    "  • Examples: os_007 Disk_rd_ios (31/31 anomalous, longest run 31), os_020 Disk_io_util (31/31, run 31), redis_001 redis_load (31/31, run 31),\n"
    "    db_001 (Hang, On_Off_State, Row_Lock, SctRead_Per_Sec) each 30/30 anomalous with longest run 30.\n"
    "- Anomaly coverage values above 1.0 appear because both high and low conditions were counted when p95 == p5 == observed value (so count_high and count_low both equal group_count).\n"
    "- Earliest anomaly timestamps for flagged groups are uniformly 2020-05-30 05:00:00+08:00.\n"
    "- Component summary (top examples): many OS components have large numbers of faulty KPIs (e.g., os_006: 51 KPIs, total anomaly points 952; os_012: 51 KPIs, total anomaly points 679; os_007: 50 KPIs, total anomaly points 918).\n"
    "- Overall counts (reference): total metric groups checked = 2060; metric groups with in-window anomalies = 2035.\n"
    "Notes: Full intermediate DataFrames remain in the kernel (metric_group_summary, component_summary_top, etc.) for drill-down if needed."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id         kpi_name  group_count_in_window  count_high  count_low  anomaly_coverage          earliest_anom_ts  longest_consecutive_anom_len  max_abs_deviation
0      os_007      Disk_rd_ios                     31          31         31               2.0 2020-05-30 05:00:00+08:00                            31             0.0000
1      os_020     Disk_io_util                     31          31         31               2.0 2020-05-30 05:00:00+08:00                            31             0.0000
2   redis_001       redis_load                     31          31         31               2.0 2020-05-30 05:00:00+08:00                            31             0.0000
3     osb_001  app.succee_rate                     31          27          4               1.0 2020-05-30 05:00:00+08:00                            31             0.0721
4      db_001             Hang                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
5      db_001     On_Off_State                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
6      db_001         Row_Lock                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
7      db_001  SctRead_Per_Sec                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
8      db_002             AIOS                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
9      db_002             Hang                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
10     db_002        MEM_Total                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
11     db_002     On_Off_State                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
12     db_002         Row_Lock                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
13     db_002  SctRead_Per_Sec                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
14     db_003             Hang                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
15     db_003     On_Off_State                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
16     db_003         Row_Lock                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
17     db_004             Hang                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
18     db_004        MEM_Total                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000
19     db_004     On_Off_State                     30          30         30               2.0 2020-05-30 05:00:00+08:00                            30             0.0000,    cmdb_id  faulty_kpi_count          earliest_anom_ts  max_longest_consecutive_anom_len  max_abs_deviation  total_anomaly_points
27  os_006                51 2020-05-30 05:00:00+08:00                                30           3.000000                   952
32  os_012                51 2020-05-30 05:00:00+08:00                                30          44.207000                   679
34  os_014                51 2020-05-30 05:00:00+08:00                                30           4.950000                   785
35  os_015                51 2020-05-30 05:00:00+08:00                                30          35.200000                   775
36  os_016                51 2020-05-30 05:00:00+08:00                                30           7.700000                   804
37  os_017                51 2020-05-30 05:00:00+08:00                                30          30.100154                   692
40  os_021                51 2020-05-30 05:00:00+08:00                                30       33553.483041                   752
0   os_007                50 2020-05-30 05:00:00+08:00                                31           5.900000                   918
1   os_020                50 2020-05-30 05:00:00+08:00                                31         106.900000                   593
22  os_001                50 2020-05-30 05:00:00+08:00                                30          12.100000                   765
23  os_002                50 2020-05-30 05:00:00+08:00                                30           6.000000                   812
25  os_004                50 2020-05-30 05:00:00+08:00                                30          63.250000                   601
28  os_008                50 2020-05-30 05:00:00+08:00                                30           2.000000                   912
29  os_009                50 2020-05-30 05:00:00+08:00                                30          16.800000                   741
30  os_010                50 2020-05-30 05:00:00+08:00                                30           3.900000                   833
31  os_011                50 2020-05-30 05:00:00+08:00                                30           4.000000                   717
33  os_013                50 2020-05-30 05:00:00+08:00                                30          19.997500                   670
38  os_018                50 2020-05-30 05:00:00+08:00                                30        3887.400000                   608
24  os_003                49 2020-05-30 05:00:00+08:00                                30         135.100000                   526
39  os_019                49 2020-05-30 05:00:00+08:00                                30         165.602500                   641, 2060, 2035)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: comp_agg (component summary) and metric_group_summary (per-group summary)
# Fallbacks if different names exist
try:
    comp_df = comp_agg
except NameError:
    try:
        comp_df = component_summary
    except NameError:
        try:
            comp_df = component_summary_top
        except NameError:
            raise NameError("Component summary not found in kernel (expected comp_agg or component_summary).")

try:
    group_df = metric_group_summary
except NameError:
    try:
        group_df = metric_group_summary_top
    except NameError:
        raise NameError("Metric group summary not found in kernel (expected metric_group_summary).")

# Ensure columns we need exist
required_comp_cols = {'cmdb_id','faulty_kpi_count','total_anomaly_points','max_longest_consecutive_anom_len','max_abs_deviation','earliest_anom_ts'}
missing = required_comp_cols - set(comp_df.columns)
if missing:
    # Try to rename if similar names present from earlier runs
    rename_map = {}
    if 'max_longest_consecutive_anom_len' not in comp_df.columns and 'max_longest_consecutive_anom_len' in comp_df.columns:
        rename_map['max_longest_consecutive_anom_len'] = 'max_longest_consecutive_anom_len'
    # if still missing, proceed but will raise
    if rename_map:
        comp_df = comp_df.rename(columns=rename_map)

# 1) Select top 8 components by total_anomaly_points desc
comp_sorted = comp_df.sort_values('total_anomaly_points', ascending=False).reset_index(drop=True)
top_components = comp_sorted.head(8).copy()

# Select and order requested columns for compact output
top_components = top_components.loc[:, ['cmdb_id','faulty_kpi_count','total_anomaly_points','max_longest_consecutive_anom_len','max_abs_deviation','earliest_anom_ts']]

# Ensure earliest_anom_ts is timezone-aware (Asia/Shanghai) for display
def ensure_tz(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, pd.Timestamp):
        if x.tzinfo is None:
            return x.tz_localize(tz)
        else:
            return x.tz_convert(tz)
    # try to parse numeric epoch
    try:
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)
    except Exception:
        return pd.NaT

top_components['earliest_anom_ts'] = top_components['earliest_anom_ts'].apply(ensure_tz)

# 2) For each top component, select up to top 6 faulty KPIs by longest_consecutive_anom_len desc, then max_abs_deviation desc
kpi_rows = []
for cmdb in top_components['cmdb_id'].tolist():
    dfc = group_df[group_df['cmdb_id'] == cmdb].copy()
    if dfc.empty:
        continue
    # Ensure required columns exist in group_df
    # Expected: kpi_name, group_count_in_window, anomaly_coverage, longest_consecutive_anom_len, max_abs_deviation, earliest_anom_ts
    sort_cols = []
    if 'longest_consecutive_anom_len' in dfc.columns:
        sort_cols.append('longest_consecutive_anom_len')
    if 'max_abs_deviation' in dfc.columns:
        sort_cols.append('max_abs_deviation')
    if not sort_cols:
        # fallback sort by anomaly_coverage then group_count_in_window
        sort_cols = ['anomaly_coverage','group_count_in_window']
    dfc_sorted = dfc.sort_values(by=sort_cols, ascending=[False]*len(sort_cols)).head(6)
    # Ensure earliest_anom_ts tz-aware
    if 'earliest_anom_ts' in dfc_sorted.columns:
        dfc_sorted['earliest_anom_ts'] = dfc_sorted['earliest_anom_ts'].apply(ensure_tz)
    # Select requested columns and append
    selected_cols = ['cmdb_id','kpi_name','group_count_in_window','anomaly_coverage','longest_consecutive_anom_len','max_abs_deviation','earliest_anom_ts']
    # Some columns might be missing; align
    for c in selected_cols:
        if c not in dfc_sorted.columns:
            dfc_sorted[c] = pd.NA
    kpi_rows.append(dfc_sorted[selected_cols])

if kpi_rows:
    top_component_kpis = pd.concat(kpi_rows, ignore_index=True)
else:
    top_component_kpis = pd.DataFrame(columns=['cmdb_id','kpi_name','group_count_in_window','anomaly_coverage','longest_consecutive_anom_len','max_abs_deviation','earliest_anom_ts'])

# Keep outputs compact: limit KPI rows to at most 8*6 (already ensured) and components to 8
top_component_kpis = top_component_kpis.head(8*6)
top_components = top_components.head(8)

# Store variables in kernel for next steps (already named)
top_components, top_component_kpis

```
Out[4]:
```
summary = (
    "Top-8 faulty components by total anomaly points (May 30 05:00–05:30 UTC+8):\n"
    "1) db_006 — 46 faulty KPIs, 1040 total anomaly points, max run 30 min, max deviation 455.39\n"
    "2) redis_008 — 22 faulty KPIs, 1022 points, max run 30 min, max deviation 5.80\n"
    "3) redis_012 — 22 faulty KPIs, 964 points, max run 30 min, max deviation 5.90\n"
    "4) os_006 — 51 faulty KPIs, 952 points, max run 30 min, max deviation 3.00\n"
    "5) redis_004 — 22 faulty KPIs, 951 points, max run 30 min, max deviation 6.66\n"
    "6) db_011 — 46 faulty KPIs, 928 points, max run 30 min, very large max deviation (~28575.13)\n"
    "7) db_010 — 46 faulty KPIs, 920 points, max run 30 min, very large max deviation (~36952.10)\n"
    "8) os_007 — 50 faulty KPIs, 918 points, max run 31 min, max deviation 5.90\n\n"
    "Top faulty KPIs (up to 6 per component, examples):\n"
    "- db_006: Physical_Read_Per_Sec, Login_Per_Sec, Hang, MEM_Total, MEM_real_util (each shows full-window consecutive anomalies, longest run 30)\n"
    "- redis_008 / redis_012 / redis_004: multiple redis KPIs show long runs (30 min) and moderate deviations (≈5–6)\n"
    "- os_006 / os_007: many OS I/O and ping-related KPIs (e.g., os_007: Disk_await, Agent_ping, Disk_io_util, Disk_wr_ios, ICMP_ping) with full-window runs (30–31 min)\n"
    "- db_010 / db_011: many DB KPIs with long runs and very large max deviations (tens of thousands), indicating major outliers on those KPIs\n\n"
    "General observations:\n"
    "- Most flagged KPIs for these top components have anomaly coverage ≈1.0–2.0 (many in-window samples are anomalous). Some coverage >1 occurs because both high and low conditions were counted when p95==p5 or values equal both thresholds.\n"
    "- Longest consecutive anomalous runs commonly span the entire incident window (30 or 31 minutes), and earliest anomaly timestamps for these groups are 2020-05-30 05:00:00+08:00.\n\n"
    "Reference counts: total metric groups checked = 2060; metric groups with in-window anomalies = 2035.\n"
    "Stored kernel variables for follow-up: top_components (component table) and top_component_kpis (per-KPI table)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  faulty_kpi_count  total_anomaly_points  max_longest_consecutive_anom_len  max_abs_deviation          earliest_anom_ts
0     db_006                46                  1040                                30            455.390 2020-05-30 05:00:00+08:00
1  redis_008                22                  1022                                30              5.800 2020-05-30 05:00:00+08:00
2  redis_012                22                   964                                30              5.900 2020-05-30 05:00:00+08:00
3     os_006                51                   952                                30              3.000 2020-05-30 05:00:00+08:00
4  redis_004                22                   951                                30              6.664 2020-05-30 05:00:00+08:00
5     db_011                46                   928                                30          28575.125 2020-05-30 05:00:00+08:00
6     db_010                46                   920                                30          36952.095 2020-05-30 05:00:00+08:00
7     os_007                50                   918                                31              5.900 2020-05-30 05:00:00+08:00,    cmdb_id               kpi_name  group_count_in_window  anomaly_coverage  longest_consecutive_anom_len  max_abs_deviation          earliest_anom_ts
0   db_006  Physical_Read_Per_Sec                     30               1.0                            30              3.700 2020-05-30 05:00:00+08:00
1   db_006          Login_Per_Sec                     30               1.0                            30              0.100 2020-05-30 05:00:00+08:00
2   db_006                   Hang                     30               2.0                            30              0.000 2020-05-30 05:00:00+08:00
3   db_006              MEM_Total                     30               2.0                            30              0.000 2020-05-30 05:00:00+08:00
4   db_006          MEM_real_util                     30               2.0                            30              0.000 2020-05-30 05:00:00+08:00
..     ...                    ...                    ...               ...                           ...                ...                       ...
43  os_007             Disk_await                     30               1.0                            30              0.152 2020-05-30 05:00:00+08:00
44  os_007             Agent_ping                     30               2.0                            30              0.000 2020-05-30 05:00:00+08:00
45  os_007           Disk_io_util                     30               2.0                            30              0.000 2020-05-30 05:00:00+08:00
46  os_007            Disk_wr_ios                     30               2.0                            30              0.000 2020-05-30 05:00:00+08:00
47  os_007              ICMP_ping                     30               2.0                            30              0.000 2020-05-30 05:00:00+08:00

[48 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: trace_df, trace_thresholds, trace_df_window (windowed by May 30 05:00-05:30 UTC+8)
# If trace_df_window missing, recreate it
try:
    trace_df_window
except NameError:
    start_local = pd.Timestamp('2020-05-30 05:00:00', tz=tz)
    end_local   = pd.Timestamp('2020-05-30 05:30:00', tz=tz)
    start_epoch = int(start_local.tz_convert('UTC').timestamp())
    end_epoch   = int(end_local.tz_convert('UTC').timestamp())
    trace_df_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()

# Ensure trace_thresholds exists (global p95/p5 per group)
try:
    trace_thresholds
except NameError:
    trace_thresholds = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    )

# Identify matching groups from full trace_thresholds (searching full series as requested)
substrs = ['to_db_010','to_db_011','from_db_010','from_db_011']
pattern = '|'.join(substrs)
matching_groups = trace_thresholds[
    (trace_thresholds['cmdb_id'].isin(['db_010','db_011'])) |
    (trace_thresholds['trace_name'].str.contains(pattern, na=False))
].copy()

matching_count = matching_groups.shape[0]

# Merge matching_groups with window rows (left join so groups with zero-window rows are retained)
merged = matching_groups.merge(trace_df_window, on=['cmdb_id','trace_name'], how='left', suffixes=('_thr','_win'))

# Flags for anomalies using global thresholds (p95/p5 from matching_groups)
merged['is_high'] = merged['value'] >= merged['p95']
merged['is_low']  = merged['value'] <= merged['p5']
merged['is_anom'] = merged['is_high'] | merged['is_low']

# Aggregate per group: group_count_in_window, count_high, count_low
agg_counts = (
    merged
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        group_count_in_window=('value','count'),
        count_high=('is_high','sum'),
        count_low=('is_low','sum'),
        p95=('p95','first'),
        p5=('p5','first')
    )
)

# earliest_anom_ts: min timestamp among anomaly rows (if none -> NaT)
anom_rows = merged[merged['is_anom'] & merged['timestamp'].notna()]
if not anom_rows.empty:
    earliest = anom_rows.groupby(['cmdb_id','trace_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anom_ts_raw'})
else:
    earliest = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anom_ts_raw'])

agg = agg_counts.merge(earliest, on=['cmdb_id','trace_name'], how='left')

# Convert earliest_anom_ts_raw (epoch seconds) to Asia/Shanghai tz-aware timestamps
def to_local_ts(x):
    if pd.isna(x):
        return pd.NaT
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)

agg['earliest_anom_ts'] = agg['earliest_anom_ts_raw'].apply(to_local_ts)
agg.drop(columns=['earliest_anom_ts_raw'], inplace=True)

# Compute how many matching groups have any in-window anomalies
agg['total_anoms'] = agg['count_high'].fillna(0).astype(int) + agg['count_low'].fillna(0).astype(int)
matching_with_anoms_count = int((agg['total_anoms'] > 0).sum())

# Final compact table: select requested columns and sort by total_anoms desc then earliest_anom_ts asc
trace_db_candidates = (
    agg
    .loc[:, ['cmdb_id','trace_name','group_count_in_window','count_high','count_low','earliest_anom_ts','p95','p5']]
    .fillna({'group_count_in_window':0,'count_high':0,'count_low':0})
)

trace_db_candidates = trace_db_candidates.assign(total_anoms=(trace_db_candidates['count_high'].astype(int) + trace_db_candidates['count_low'].astype(int)))
trace_db_candidates = trace_db_candidates.sort_values(['total_anoms','earliest_anom_ts'], ascending=[False, True]).head(20)
trace_db_candidates = trace_db_candidates.drop(columns=['total_anoms'])

# Store result for next steps
# trace_db_candidates (DataFrame), matching_count (int), matching_with_anoms_count (int)
trace_db_candidates, matching_count, matching_with_anoms_count

```
Out[5]:
```
summary = (
    "Search results for traces referencing db_010 or db_011 (by cmdb_id or trace_name containing "
    "'to_db_010','to_db_011','from_db_010','from_db_011'):\n"
    "- No matching trace groups were found in trace_df (matching_count = 0).\n"
    "- Consequently, there are no in-window anomalies for such traces (matching_with_anoms_count = 0).\n"
    "- The returned DataFrame trace_db_candidates is empty.\n\n"
    "Stored kernel variables for follow-up: trace_db_candidates (empty DataFrame), matching_count (0), matching_with_anoms_count (0)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, group_count_in_window, count_high, count_low, earliest_anom_ts, p95, p5]
Index: [], 0, 0)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables if available
try:
    group_df = metric_group_summary
except NameError:
    try:
        group_df = metric_anoms
    except NameError:
        raise NameError("Neither metric_group_summary nor metric_anoms found in kernel.")

try:
    thresholds = metric_thresholds
except NameError:
    # compute global thresholds if missing (should not happen per prior steps)
    thresholds = (
        metric_df
        .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    )

# Filter for db_010 and db_011 within the in-window-flagged groups (group_df contains only in-window-flagged groups)
db_filter = group_df[group_df['cmdb_id'].isin(['db_010','db_011'])].copy()

# Merge full-series thresholds (p95/p5)
db010_db011_kpis = db_filter.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Ensure earliest_anom_ts is timezone-aware Asia/Shanghai
def ensure_tz(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, pd.Timestamp):
        if x.tzinfo is None:
            return x.tz_localize(tz)
        else:
            return x.tz_convert(tz)
    try:
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)
    except Exception:
        return pd.NaT

if 'earliest_anom_ts' in db010_db011_kpis.columns:
    db010_db011_kpis['earliest_anom_ts'] = db010_db011_kpis['earliest_anom_ts'].apply(ensure_tz)

# Select and order requested columns; ensure columns exist
cols = ['cmdb_id','kpi_name','group_count_in_window','count_high','count_low',
        'anomaly_coverage','longest_consecutive_anom_len','max_abs_deviation','earliest_anom_ts','p95','p5']
for c in cols:
    if c not in db010_db011_kpis.columns:
        db010_db011_kpis[c] = pd.NA

db010_db011_kpis = db010_db011_kpis.loc[:, cols]

# Sort by longest_consecutive_anom_len desc, then max_abs_deviation desc, limit 50
db010_db011_kpis = db010_db011_kpis.sort_values(['longest_consecutive_anom_len','max_abs_deviation'], ascending=[False, False]).head(50).reset_index(drop=True)

# Counts per component among these in-window-flagged KPI groups
count_db010 = int((db_filter['cmdb_id']=='db_010').sum())
count_db011 = int((db_filter['cmdb_id']=='db_011').sum())
db010_db011_counts = pd.DataFrame([{'cmdb_id':'db_010','kpi_count':count_db010},
                                   {'cmdb_id':'db_011','kpi_count':count_db011}])

# Final compact outputs (stored in kernel variables for next step)
db010_db011_kpis, db010_db011_counts

```
Out[6]:
```
Summary of db_010 / db_011 KPI anomalies (May 30, 2020 05:00–05:30 UTC+8):

- Counts:
  - db_010: 46 in-window flagged KPI groups.
  - db_011: 46 in-window flagged KPI groups.
  (Stored as db010_db011_counts.)

- General pattern:
  - Many KPIs for both components have nearly all in-window samples flagged as anomalous, with longest consecutive anomalous runs spanning the whole incident window (30 minutes) and earliest anomaly timestamps at 2020-05-30 05:00:00+08:00.
  - Anomaly coverage is often ~1.0 (all samples anomalous) or >1.0 when both high and low conditions are counted.

- Notable per-KPI examples (from the compact per-KPI table db010_db011_kpis):
  - db_010: Physical_Read_Per_Sec (30 samples, 3 high / 27 low, longest run 30, p95=117.7, p5=0), Login_Per_Sec, ACS, Redo_Per_Sec (has very large max_abs_deviation ≈ 36952.1), Logic_Read_Per_Sec (large p95/p5 values).
  - db_011: TPS_Per_Sec (30 samples, 30 high / 29 low, anomaly_coverage ≈1.97, longest run 30), User_Commit, and many other KPIs with full-window anomalous runs.

- Stored outputs for follow-up:
  - db010_db011_kpis — compact per-KPI table (includes group counts, counts of highs/lows, coverage, longest run, max_abs_deviation, earliest anomaly ts, and p95/p5).
  - db010_db011_counts — counts per component (db_010 and db_011).

If you want, I can show the top N rows of db010_db011_kpis for direct inspection (e.g., df.head(20)).

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id               kpi_name  group_count_in_window  count_high  count_low  anomaly_coverage  longest_consecutive_anom_len  max_abs_deviation          earliest_anom_ts          p95          p5
0   db_010  Physical_Read_Per_Sec                     30           3         27          1.000000                            30           2.000000 2020-05-30 05:00:00+08:00   117.700000     0.00000
1   db_011            TPS_Per_Sec                     30          30         29          1.966667                            30           0.400000 2020-05-30 05:00:00+08:00     0.000000     0.00000
2   db_011            User_Commit                     30           1         29          1.000000                            30           0.377500 2020-05-30 05:00:00+08:00     0.002500     0.00000
3   db_010          Login_Per_Sec                     30           3         27          1.000000                            30           0.100000 2020-05-30 05:00:00+08:00     0.200000     0.10000
4   db_010                    ACS                     30           2         28          1.000000                            30           0.009500 2020-05-30 05:00:00+08:00     0.010500     0.01000
..     ...                    ...                    ...         ...        ...               ...                           ...                ...                       ...          ...         ...
45  db_011    DFParaWrite_Per_Sec                     30           2          6          0.266667                             2           0.010000 2020-05-30 05:04:00+08:00     0.080000     0.04000
46  db_011                    ACS                     30           1          2          0.100000                             2           0.009500 2020-05-30 05:02:00+08:00     0.010500     0.00000
47  db_010               MEM_Used                     30           0          3          0.100000                             2           0.005359 2020-05-30 05:01:00+08:00   325.793063   325.71054
48  db_010           Redo_Per_Sec                     30           2          1          0.100000                             1       36952.095000 2020-05-30 05:01:00+08:00   255.005000    92.85500
49  db_010     Logic_Read_Per_Sec                     30           2          0          0.066667                             1         111.980000 2020-05-30 05:18:00+08:00  6328.520000  1030.56500

[50 rows x 11 columns],   cmdb_id  kpi_count
0  db_010         46
1  db_011         46)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: metric_df, metric_thresholds, metric_df_window if present; otherwise recreate window
kpis = [
    "Redo_Per_Sec", "Physical_Read_Per_Sec", "Logic_Read_Per_Sec",
    "TPS_Per_Sec", "Login_Per_Sec", "Hang", "On_Off_State", "MEM_Used"
]

# Ensure global thresholds exist
try:
    metric_thresholds
except NameError:
    metric_thresholds = (
        metric_df
        .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    )

# Ensure windowed metric_df exists (May 30, 2020 05:00-05:30 UTC+8 inclusive)
try:
    metric_df_window
except NameError:
    start_local = pd.Timestamp('2020-05-30 05:00:00', tz=tz)
    end_local   = pd.Timestamp('2020-05-30 05:30:00', tz=tz)
    start_epoch = int(start_local.tz_convert('UTC').timestamp())
    end_epoch   = int(end_local.tz_convert('UTC').timestamp())
    metric_df_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()

# Filter window rows for db_010 and target KPIs
db010_window = metric_df_window[(metric_df_window['cmdb_id'] == 'db_010') & (metric_df_window['kpi_name'].isin(kpis))].copy()

# Helper to convert epoch -> localized tz timestamp
def epoch_to_local_ts(x):
    if pd.isna(x):
        return pd.NaT
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)

rows = []
# For each KPI, compute requested stats
for k in kpis:
    grp = db010_window[db010_window['kpi_name'] == k].copy()
    group_count_in_window = int(grp.shape[0])
    if group_count_in_window > 0:
        vmin = float(grp['value'].min())
        v50  = float(grp['value'].quantile(0.50))
        v95w = float(grp['value'].quantile(0.95))
        vmax = float(grp['value'].max())
        # earliest timestamp where value == max (within window)
        ts_max_rows = grp[grp['value'] == vmax]
        if not ts_max_rows.empty:
            earliest_ts_of_max = epoch_to_local_ts(int(ts_max_rows['timestamp'].min()))
        else:
            earliest_ts_of_max = pd.NaT
    else:
        vmin = v50 = v95w = vmax = float('nan')
        earliest_ts_of_max = pd.NaT

    # full-series thresholds from metric_thresholds (must be computed prior)
    thr_row = metric_thresholds[(metric_thresholds['cmdb_id']=='db_010') & (metric_thresholds['kpi_name']==k)]
    if not thr_row.empty:
        p95_full = float(thr_row['p95'].iloc[0])
        p5_full  = float(thr_row['p5'].iloc[0])
    else:
        p95_full = float('nan')
        p5_full  = float('nan')

    # count_high and count_low using full-series thresholds
    if group_count_in_window > 0 and (not pd.isna(p95_full)):
        count_high = int((grp['value'] >= p95_full).sum())
        count_low  = int((grp['value'] <= p5_full).sum())
        # compute max absolute deviation among anomaly rows
        # For highs: abs(value - p95_full); for lows: abs(value - p5_full)
        devs = []
        highs = grp[grp['value'] >= p95_full]
        if not highs.empty:
            devs.append((highs['value'] - p95_full).abs().max())
        lows = grp[grp['value'] <= p5_full]
        if not lows.empty:
            devs.append((lows['value'] - p5_full).abs().max())
        max_abs_deviation = float(max(devs)) if devs else 0.0
        # for identifying earliest timestamp of the row achieving max_abs_deviation:
        if devs:
            # compute per-row deviation and find earliest timestamp where it equals max_abs_deviation
            grp['dev'] = 0.0
            if not highs.empty:
                grp.loc[highs.index, 'dev'] = (grp.loc[highs.index, 'value'] - p95_full).abs()
            if not lows.empty:
                grp.loc[lows.index, 'dev'] = (grp.loc[lows.index, 'value'] - p5_full).abs()
            max_dev_val = float(max_abs_deviation)
            candidate_rows = grp[grp['dev'] == max_dev_val]
            if not candidate_rows.empty:
                earliest_max_dev_ts = epoch_to_local_ts(int(candidate_rows['timestamp'].min()))
            else:
                earliest_max_dev_ts = pd.NaT
        else:
            earliest_max_dev_ts = pd.NaT
    else:
        count_high = 0
        count_low = 0
        max_abs_deviation = float('nan')
        earliest_max_dev_ts = pd.NaT

    rows.append({
        'cmdb_id': 'db_010',
        'kpi_name': k,
        'group_count_in_window': group_count_in_window,
        'min_value': vmin,
        'p50_value': v50,
        'p95_value_window': v95w,
        'max_value': vmax,
        'earliest_timestamp_of_max': earliest_ts_of_max,
        'count_high': count_high,
        'count_low': count_low,
        'p95_full': p95_full,
        'p5_full': p5_full,
        'max_abs_deviation': max_abs_deviation,
        'earliest_max_deviation_ts': earliest_max_dev_ts
    })

db010_key_kpi_stats = pd.DataFrame(rows)

# Identify the single KPI with the largest absolute deviation (max_abs_deviation) and its earliest anomalous timestamp
if not db010_key_kpi_stats['max_abs_deviation'].isna().all():
    idx = db010_key_kpi_stats['max_abs_deviation'].idxmax()
    largest_kpi = db010_key_kpi_stats.loc[idx, 'kpi_name']
    largest_dev_value = db010_key_kpi_stats.loc[idx, 'max_abs_deviation']
    largest_dev_earliest_ts = db010_key_kpi_stats.loc[idx, 'earliest_max_deviation_ts']
    largest_dev_summary = pd.DataFrame([{
        'kpi_name': largest_kpi,
        'max_abs_deviation': float(largest_dev_value),
        'earliest_max_deviation_ts': largest_dev_earliest_ts
    }])
else:
    largest_dev_summary = pd.DataFrame([{'kpi_name': pd.NA, 'max_abs_deviation': pd.NA, 'earliest_max_deviation_ts': pd.NaT}])

# Keep outputs compact: show the 8-row table and the single-row summary
db010_key_kpi_stats, largest_dev_summary

```
Out[7]:
```
Summary of db_010 in-window stats (May 30, 2020 05:00–05:30 UTC+8):

All 8 KPIs had 30 samples in the incident window.

Per-KPI highlights (group_count_in_window = 30 for all):
- Redo_Per_Sec
  - min 81.5, p50 141.35, window p95 1017.435, max 37207.10 (earliest at 2020-05-30 05:01:00+08:00)
  - count_high = 2, count_low = 1
  - full-series p95 = 255.005, p5 = 92.855
  - max_abs_deviation = 36952.095 (earliest at 2020-05-30 05:01:00+08:00) — the single largest deviation among the 8 KPIs

- Physical_Read_Per_Sec
  - min 0, p50 0, window p95 117.7, max 119.7 (earliest 05:28)
  - count_high = 3, count_low = 27
  - full-series p95 = 117.7, p5 = 0
  - max_abs_deviation = 2.0 (earliest 05:28)

- Logic_Read_Per_Sec
  - min 1049, p50 1076.95, window p95 6328.72, max 6440.50 (earliest 05:28)
  - count_high = 2, count_low = 0
  - full-series p95 = 6328.52, p5 = 1030.565
  - max_abs_deviation = 111.98 (earliest 05:28)

- TPS_Per_Sec
  - min 36.8, p50 41.55, window p95 42.9, max 44.1 (earliest 05:19)
  - count_high = 1, count_low = 0
  - full-series p95 = 43.61, p5 = 36.56
  - max_abs_deviation = 0.49 (earliest 05:19)

- Login_Per_Sec
  - min 0.1, p50 0.1, window p95 0.2, max 0.3 (earliest 05:28)
  - count_high = 3, count_low = 27
  - full-series p95 = 0.2, p5 = 0.1
  - max_abs_deviation = 0.1 (earliest 05:28)

- Hang
  - constant 1.0 (min/p50/p95/max all 1.0), earliest max 05:00
  - count_high = 30, count_low = 30 (p95==p5==1.0)
  - full-series p95 = 1.0, p5 = 1.0
  - max_abs_deviation = 0.0

- On_Off_State
  - constant 1.0 (same behavior as Hang)
  - count_high = 30, count_low = 30
  - full-series p95 = 1.0, p5 = 1.0
  - max_abs_deviation = 0.0

- MEM_Used
  - min 325.705, p50 325.7459, window p95 325.7769, max 325.79143 (earliest 05:28)
  - count_high = 0, count_low = 3
  - full-series p95 = 325.793063, p5 = 325.71054
  - max_abs_deviation = 0.005359 (earliest 05:24)

Single largest absolute deviation:
- KPI: Redo_Per_Sec
- max_abs_deviation: 36952.095
- earliest anomalous timestamp for that deviation: 2020-05-30 05:01:00+08:00

Stored kernel variable for follow-up: db010_key_kpi_stats (table with the 8 KPI rows and the computed fields).

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id               kpi_name  group_count_in_window    min_value   p50_value  p95_value_window    max_value earliest_timestamp_of_max  count_high  count_low     p95_full     p5_full  max_abs_deviation earliest_max_deviation_ts
0  db_010           Redo_Per_Sec                     30    81.500000   141.35000       1017.435000  37207.10000 2020-05-30 05:01:00+08:00           2          1   255.005000    92.85500       36952.095000 2020-05-30 05:01:00+08:00
1  db_010  Physical_Read_Per_Sec                     30     0.000000     0.00000        117.700000    119.70000 2020-05-30 05:28:00+08:00           3         27   117.700000     0.00000           2.000000 2020-05-30 05:28:00+08:00
2  db_010     Logic_Read_Per_Sec                     30  1049.000000  1076.95000       6328.720000   6440.50000 2020-05-30 05:28:00+08:00           2          0  6328.520000  1030.56500         111.980000 2020-05-30 05:28:00+08:00
3  db_010            TPS_Per_Sec                     30    36.800000    41.55000         42.900000     44.10000 2020-05-30 05:19:00+08:00           1          0    43.610000    36.56000           0.490000 2020-05-30 05:19:00+08:00
4  db_010          Login_Per_Sec                     30     0.100000     0.10000          0.200000      0.30000 2020-05-30 05:28:00+08:00           3         27     0.200000     0.10000           0.100000 2020-05-30 05:28:00+08:00
5  db_010                   Hang                     30     1.000000     1.00000          1.000000      1.00000 2020-05-30 05:00:00+08:00          30         30     1.000000     1.00000           0.000000 2020-05-30 05:00:00+08:00
6  db_010           On_Off_State                     30     1.000000     1.00000          1.000000      1.00000 2020-05-30 05:00:00+08:00          30         30     1.000000     1.00000           0.000000 2020-05-30 05:00:00+08:00
7  db_010               MEM_Used                     30   325.705181   325.74589        325.776888    325.79143 2020-05-30 05:28:00+08:00           0          3   325.793063   325.71054           0.005359 2020-05-30 05:24:00+08:00,        kpi_name  max_abs_deviation earliest_max_deviation_ts
0  Redo_Per_Sec          36952.095 2020-05-30 05:01:00+08:00)```
```